## Retrieval Optimisation Techniques

### 1) Top-K Tradeoff

In [1]:
# We pick the most relevant documents/chunks

In [2]:
# Case 1

top_k = 1 # very low

# Pros 
#     High Precision
#     Fast processing as only one chunk
#     Cost effectvie
# Cons 
#     May miss out required info (context)

In [3]:
# Case 2

top_k = 10 # very large

# Pros 
#     More infor (context)
# Cons 
#     Low Precision
#     High Cost
#     May contain some irrelevant info
#     Slow Processings

In [4]:
# Case 3

top_k = 5 # medium

#### Debugging Top-K

In [5]:
import os
import chromadb
from pathlib import Path
from openai import OpenAI

In [6]:
def load_documents(data_folder):
    documents= []
    doc_ids = []

    for file in os.listdir(data_folder):
        file_path = os.path.join(data_folder, file)

        with open(file_path, 'r') as f:
            text = f.read()
        
        documents.append(text)
        doc_ids.append(file.removesuffix('.txt'))

    return [documents, doc_ids]

In [7]:
def get_embeddings_from_llm(llm_client, text):
    response = llm_client.embeddings.create(
        model='text-embedding-3-small',
        input=text
    )

    return response.data[0].embedding

In [8]:
def store_embeddings(llm_client, documents):
    embeddings = []
    for doc in documents:
        emb = get_embeddings_from_llm(llm_client=llm_client, text=doc)
        embeddings.append(emb)

    return embeddings

In [19]:
def create_collection(db_client):
    collection = db_client.create_collection(name='company_policy_info_topk')
    return collection

In [10]:
def add_documents_to_collection(collection, documents, embeddings, ids):
    collection.add(
        documents=documents,
        embeddings=embeddings,
        ids=ids
    )

In [11]:
def get_context(collection, llm_client, query, top_k):
    query_emb = get_embeddings_from_llm(llm_client=llm_client, text=query)

    db_results = collection.query(
        query_embeddings = query_emb,
        n_results=top_k
    )

    docs = db_results['documents'][0]
    distances = db_results['distances'][0]

    for i in range(len(docs)):
        print(f'Rank: {i+1}')
        print(f'Distance: {distances[i]}')
        print(docs[i])
        print("---------------------------\n")

    return '\n'.join(docs)

In [15]:
def create_prompt(context, query):
    prompt=f'''Answer the query based upon the given context only.
CONTEXT:
{context}
QUERY:
{query}
    '''

    return prompt

In [16]:
def generator(llm_client, prompt):
    response = llm_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}]
    )

    return response.choices[0].message.content

In [14]:
# Create OpenAI & ChromaDB clients

openai_client = OpenAI(api_key=os.getenv('OPENAI_SECRET_KEY'))
db_client = chromadb.PersistentClient(path='./topk_chroma_db')

In [18]:
documents, doc_ids = load_documents(data_folder='data')
documents, doc_ids

(['AcmeTech Solutions Code of Conduct\n\nEmployees must maintain professional behavior at all times.\n\nHarassment, discrimination, or unethical conduct will not be tolerated.',
  'AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.\n\nAll systems must be locked when unattended.\n\nSensitive data must be stored securely and encrypted where appropriate.',
  'AcmeTech Solutions Leave Policy\n\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\n\nLeave requests must be submitted at least two weeks in advance.\n\nUnused leave may not be carried over to the next year unless approved.',
  'AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.\n\nRemote work must be approved by the employeeâ€™s manager.\n\nEmployees must ensure a secure and productive work environment while working remotely.'],
 ['code_of_conduct',
  'it_security_policy',
  'leave_policy',
  'remot

In [20]:
embeddings = store_embeddings(llm_client=openai_client, documents=documents)

In [21]:
policy_collection = create_collection(db_client=db_client)

In [22]:
add_documents_to_collection(collection=policy_collection, 
                            documents=documents,
                            embeddings=embeddings, 
                            ids=doc_ids)

In [24]:
query = 'Can employees work remotely?'

topks = [1,3,5]

for k in topks:
    print(f"\n\n*********TOPK-{k}**********")
    context = get_context(collection=policy_collection, llm_client=openai_client, query=query, top_k=k)

    prompt = create_prompt(context=context, query=query)
    print("-----PROMPT-----")
    print(prompt)

    result = generator(llm_client=openai_client, prompt=prompt)
    print("-----RESULT-----")
    print(result)



*********TOPK-1**********
Rank: 1
Distance: 0.6608433127403259
AcmeTech Solutions Remote Work Policy

Employees are allowed to work remotely up to 2 days per week.

Remote work must be approved by the employeeâ€™s manager.

Employees must ensure a secure and productive work environment while working remotely.
---------------------------

-----PROMPT-----
Answer the query based upon the given context only.
CONTEXT:
AcmeTech Solutions Remote Work Policy

Employees are allowed to work remotely up to 2 days per week.

Remote work must be approved by the employeeâ€™s manager.

Employees must ensure a secure and productive work environment while working remotely.
QUERY:
Can employees work remotely?
    
-----RESULT-----
Yes, employees can work remotely, up to 2 days per week, with manager approval.


*********TOPK-3**********
Rank: 1
Distance: 0.6608433127403259
AcmeTech Solutions Remote Work Policy

Employees are allowed to work remotely up to 2 days per week.

Remote work must be approve

In [25]:
# We cannot determine the optimal value for top-k.

### 2) Similarity Threshold

In [ ]:
# distance-L2
# Lower the distance, closer the vectors

# Similarity= 1- distance
# Higher the similarity, closer the vectors

# Cosine Similarity
# It may be between -1 to +1
# Embedding models generally give the vectors where cosine similarity is from 0 to 1

In [27]:
# Score
# 1 - Most relevant (Almost same thing)
# 0.8 - Strongly similar
# 0.5 - Moderately similar
# 0.3 - Less similarity
# 0 - No similarity

# similarity >= 0.5 -> keep the chunk/document
# similarity < 0.5 -> discard

In [28]:
query = 'What is the capital of India?'

get_context(collection=policy_collection, llm_client=openai_client, query=query, top_k=1)

Rank: 1
Distance: 1.7765328884124756
AcmeTech Solutions IT Security Policy

Employees must not share their passwords with anyone.

All systems must be locked when unattended.

Sensitive data must be stored securely and encrypted where appropriate.
---------------------------



'AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.\n\nAll systems must be locked when unattended.\n\nSensitive data must be stored securely and encrypted where appropriate.'

In [29]:
def get_context_with_threshold(collection, llm_client, query, top_k, threshold=0.5):
    query_emb = get_embeddings_from_llm(llm_client=llm_client, text=query)

    db_results = collection.query(
        query_embeddings = query_emb,
        n_results=top_k,
    )

    docs = db_results['documents'][0]
    distances = db_results['distances'][0]

    filtered_docs = []
    print('Retrieval Debug Info...\n')

    for i, (doc, distance) in enumerate(zip(docs, distances)):
        similarity = 1 - distance
        print('Rank: ', i+1)
        print('Distance: ', distance)
        print('Similarity: ', similarity)

        if similarity>=threshold:
            filtered_docs.append(doc)
            print('Status: Accepted')
        else:
            print('Status: Rejected')

    return '\n'.join(filtered_docs)

In [30]:
query = 'What is the capital of India?'

get_context_with_threshold(collection=policy_collection, llm_client=openai_client, query=query, top_k=1)

Retrieval Debug Info...

Rank:  1
Distance:  1.7765328884124756
Similarity:  -0.7765328884124756
Status: Rejected


''

In [31]:
collection = db_client.create_collection(name='threshold_test',
                                         metadata={'hnsw:space': 'cosine'})
# chromadb will use cosine similarity instead of euclidian distance.
# hnsw is the algorithm.

In [32]:
add_documents_to_collection(collection=collection, 
                            documents=documents,
                            embeddings=embeddings, 
                            ids=doc_ids)

In [35]:
query = 'Can an employee work remotely'

get_context_with_threshold(collection=collection, llm_client=openai_client, query=query, top_k=5)

Retrieval Debug Info...

Rank:  1
Distance:  0.3478114604949951
Similarity:  0.6521885395050049
Status: Accepted
Rank:  2
Distance:  0.6542878150939941
Similarity:  0.34571218490600586
Status: Rejected
Rank:  3
Distance:  0.6607774496078491
Similarity:  0.3392225503921509
Status: Rejected
Rank:  4
Distance:  0.6942541599273682
Similarity:  0.30574584007263184
Status: Rejected


'AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.\n\nRemote work must be approved by the employeeâ€™s manager.\n\nEmployees must ensure a secure and productive work environment while working remotely.'

In [36]:
query = 'What is the capital of India?'

get_context_with_threshold(collection=collection, llm_client=openai_client, query=query, top_k=5)

Retrieval Debug Info...

Rank:  1
Distance:  0.8882666826248169
Similarity:  0.1117333173751831
Status: Rejected
Rank:  2
Distance:  0.8935425877571106
Similarity:  0.1064574122428894
Status: Rejected
Rank:  3
Distance:  0.9237940907478333
Similarity:  0.07620590925216675
Status: Rejected
Rank:  4
Distance:  0.931885302066803
Similarity:  0.06811469793319702
Status: Rejected


''

In [38]:
query = 'Can an employee work remotely'

get_context_with_threshold(collection=collection, llm_client=openai_client, query=query, top_k=5, threshold=0.33)

Retrieval Debug Info...

Rank:  1
Distance:  0.3478114604949951
Similarity:  0.6521885395050049
Status: Accepted
Rank:  2
Distance:  0.6542878150939941
Similarity:  0.34571218490600586
Status: Accepted
Rank:  3
Distance:  0.6607774496078491
Similarity:  0.3392225503921509
Status: Accepted
Rank:  4
Distance:  0.6942541599273682
Similarity:  0.30574584007263184
Status: Rejected


'AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.\n\nRemote work must be approved by the employeeâ€™s manager.\n\nEmployees must ensure a secure and productive work environment while working remotely.\nAcmeTech Solutions Leave Policy\n\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\n\nLeave requests must be submitted at least two weeks in advance.\n\nUnused leave may not be carried over to the next year unless approved.\nAcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.\n\nAll systems must be locked when unattended.\n\nSensitive data must be stored securely and encrypted where appropriate.'